# Immortalite Zero — Self-Play Training (Lightning AI)

Trains on a studio GPU. Checkpoints and metrics save to **`../results`** (sibling of the repo).

Before running: upload **`results/`** and **`syzygy345/`** next to the cloned repo:

```
parent/
├── immortalite-zero/   ← this repo
├── results/            ← latest.pt, metrics.csv, …
└── syzygy345/          ← 145 .rtbw files (~378 MB)
```

**No manual config needed** — run cells top to bottom. `resume` is on by default so each session picks up from `latest.pt` in `results/`. Empty folder → starts at iter 0.

**Flat settings:** 100 MCTS sims, 128 games/iter (concurrency 128), 800 train steps, replay 200k, draw penalty **1/3** (football 3-1-0). Gate **winrates** use normal chess (draw = half point); only self-play search/training uses the high draw penalty.

**Gates:** every 20 iters, current net vs snapshot 20 iters ago (64 games, 100 sims). First gate at iter 20 vs 0.

Select a **GPU** machine in Lightning AI before running.

In [ ]:
# 1. Paths — sibling results/ and syzygy345/ (upload before running).
import os

_cwd = os.path.abspath(os.getcwd())
if os.path.isdir(os.path.join(_cwd, 'engine')):
    REPO_DIR = _cwd
elif os.path.isdir(os.path.join(_cwd, '..', 'engine')):
    REPO_DIR = os.path.abspath(os.path.join(_cwd, '..'))
else:
    raise RuntimeError(
        'Run from the repo root (immortalite-zero) or lightning-ai/ subfolder.'
    )

PARENT = os.path.dirname(REPO_DIR)
CKPT_DIR = os.path.join(PARENT, 'results')
TB_DIR = os.path.join(PARENT, 'syzygy345')

os.makedirs(CKPT_DIR, exist_ok=True)

print('repo:       ', REPO_DIR)
print('checkpoints:', CKPT_DIR)
print('syzygy:     ', TB_DIR)

if os.path.exists(os.path.join(CKPT_DIR, 'latest.pt')):
    print('resume ready: latest.pt found — cell 5 will continue from saved iteration')
else:
    print('fresh start: no latest.pt — cell 5 will begin at iter 0')

%cd {REPO_DIR}
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Lightning AI studios usually ship a CUDA torch build).
!pip install -q python-chess numpy tqdm pandas matplotlib

In [ ]:
# 3. Confirm GPU is available and choose the train preset once.
import torch

HAS_CUDA = torch.cuda.is_available()
CUDA_DEVICE = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'
IS_T4 = HAS_CUDA and ('T4' in CUDA_DEVICE)
TRAIN_PRESET = '--device cuda --gpu' if HAS_CUDA else '--device cpu --light'

print('CUDA available:', HAS_CUDA)
print('device:', CUDA_DEVICE)
print('is_t4:', IS_T4)
print('train preset:', TRAIN_PRESET)

In [ ]:
# 4. Syzygy 3-4-5 WDL — expects ../syzygy345 from manual upload.
import os

if 'TB_DIR' not in globals():
    raise RuntimeError('Run cell 1 first (paths).')

EXPECTED_RTBW = 145
if not os.path.isdir(TB_DIR):
    raise RuntimeError(
        f'Syzygy folder missing: {TB_DIR}\n'
        'Upload syzygy345/ as a sibling of the repo (145 .rtbw files).'
    )

rtbw_count = len([f for f in os.listdir(TB_DIR) if f.endswith('.rtbw')])
print(f'Syzygy -> {TB_DIR} ({rtbw_count}/{EXPECTED_RTBW} .rtbw files)')
if rtbw_count < EXPECTED_RTBW:
    raise RuntimeError(
        f'Incomplete syzygy345 upload ({rtbw_count}/{EXPECTED_RTBW}).\n'
        'Locally: python scripts/download_syzygy345.py --out syzygy345'
    )
print('Syzygy ready.')

In [ ]:
# 5. Config + train (flat settings, in-loop gates). Edit TRAIN below only.
# games == concurrency for full GPU batches; train_steps scaled with games (~5.7x reuse).
# replay_buffer/window 200k: ~6 iters at 256 games; train_steps 1600 keeps ~6x reuse.
#
# LR: held constant (lr == lr_min). When absolute gate plateaus (~15-20 iters), drop both
# ~3-5x (e.g. 2.5e-4 -> 6e-5) and re-resume. Resume from latest.pt; do not rewind.
#
# Absolute strength vs a fixed checkpoint (e.g. latest vs iter 80): use cell 6 manual gate.
import os

if 'TRAIN_PRESET' not in globals():
    raise RuntimeError('Run cell 3 first.')

TRAIN = {
    'sims': 100,
    'gate_sims': 100,
    'games': 256,
    'train_steps': 1600,
    'concurrency': 128,
    'selfplay_workers': 4,
    'replay_buffer': 200_000,
    'replay_window': 200_000,
    'draw_penalty': 1 / 3,  # football 3-1-0: draw ≈ one-third of a win
    'gate_every': 20,
    'gate_games': 512,
    'gate_exploration_moves': 20,
    'save_every': 10,
    'iterations': 1000,
    'resume': True,    # on by default — re-upload results/ with latest.pt to continue
    'resign': False,
    'lr': 2.5e-4,
    'lr_min': 2.5e-4,
    'lr_total_iters': 1000,
    'grad_clip': 10.0,
}
RESIGN_THRESHOLD = -0.90
RESIGN_PLIES = 3
RESIGN_MIN_MOVES = 20

resume_arg = ''
if TRAIN['resume']:
    resume_path = os.path.join(CKPT_DIR, 'latest.pt')
    if os.path.exists(resume_path):
        resume_arg = f'--resume {resume_path}'
    else:
        print('WARNING: resume=True but no latest.pt — starting at iter 0')

syzygy_arg = f'--syzygy-path "{TB_DIR}"'
resign_arg = ''
if TRAIN['resign']:
    resign_arg = (
        f'--resign-threshold {RESIGN_THRESHOLD} '
        f'--resign-plies {RESIGN_PLIES} '
        f'--resign-min-moves {RESIGN_MIN_MOVES}'
    )

print('TRAIN config:', TRAIN)
print(f'preset={TRAIN_PRESET} ckpt_dir={CKPT_DIR}')

!python -m engine.train --iterations {TRAIN['iterations']} {TRAIN_PRESET} --games {TRAIN['games']} --train-steps {TRAIN['train_steps']} --concurrency {TRAIN['concurrency']} --selfplay-workers {TRAIN['selfplay_workers']} --replay-buffer {TRAIN['replay_buffer']} --replay-window {TRAIN['replay_window']} --sims {TRAIN['sims']} --draw-penalty {TRAIN['draw_penalty']} {resign_arg} {syzygy_arg} --save-every {TRAIN['save_every']} --gate-every {TRAIN['gate_every']} --gate-games {TRAIN['gate_games']} --gate-sims {TRAIN['gate_sims']} --gate-exploration-moves {TRAIN['gate_exploration_moves']} --lr {TRAIN['lr']} --lr-min {TRAIN['lr_min']} --lr-total-iters {TRAIN['lr_total_iters']} --grad-clip {TRAIN['grad_clip']} --checkpoint-dir "{CKPT_DIR}" {resume_arg}

In [ ]:
# 6. Manual gate (optional) — same play_match settings as in-loop gate in cell 5:
#   gate_games, gate_sims, gate_exploration_moves, concurrency, tablebase via TRAIN.
import os
import chess.syzygy
import torch
from engine.config import Config, NetConfig
from engine.encoding import ENCODING_VERSION
from engine.network import ChessNet
from engine.sprt import ALPHA, BETA, ELO0, ELO1, sprt_verdict_label
from engine.train import play_match, _log_gate_metrics, _load_matching_state_dict

if 'TRAIN' not in globals():
    raise RuntimeError('Run cell 5 first (defines TRAIN).')

# e.g. latest vs 80 for absolute strength; or 100 vs 80 after iter 100.
CHECKPOINT_A = 20
CHECKPOINT_B = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
cfg.train.selfplay_concurrency = TRAIN['concurrency']
cfg.train.draw_penalty = TRAIN['draw_penalty']
cfg.train.syzygy_path = TB_DIR
tablebase = chess.syzygy.open_tablebase(TB_DIR) if os.path.isdir(TB_DIR) else None


def _checkpoint_iteration(checkpoint_ref, state):
    if checkpoint_ref == 'latest':
        if isinstance(state, dict):
            return int(state.get('iteration', -1))
        return -1
    try:
        return int(checkpoint_ref)
    except (TypeError, ValueError):
        return -1


def _load_gate_net(path: str) -> tuple[ChessNet, dict]:
    state = torch.load(path, map_location=DEVICE)
    enc = int(state.get('encoding_version', 1)) if isinstance(state, dict) else 1
    if enc != ENCODING_VERSION:
        raise ValueError(f'{os.path.basename(path)}: encoding {enc} != {ENCODING_VERSION}')
    net_cfg = Config().net
    if isinstance(state, dict) and 'net' in state:
        net_cfg = NetConfig(**state['net'])
    net = ChessNet(net_cfg).to(DEVICE)
    model = state['model'] if isinstance(state, dict) and 'model' in state else state
    _load_matching_state_dict(net, model, label=f'gate load {os.path.basename(path)}', verbose=False)
    net.eval()
    return net, state


# Resolve Path A
if CHECKPOINT_A == 'latest':
    path_a = os.path.join(CKPT_DIR, 'latest.pt')
    label_a = "Latest"
else:
    path_a = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_A:04d}.pt')
    label_a = f"Iter {CHECKPOINT_A}"

# Resolve Path B
if CHECKPOINT_B == 'latest':
    path_b = os.path.join(CKPT_DIR, 'latest.pt')
    label_b = "Latest"
else:
    path_b = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_B:04d}.pt')
    label_b = f"Iter {CHECKPOINT_B}"

if not os.path.exists(path_a) or not os.path.exists(path_b):
    print(f"Error: Make sure both checkpoints exist!\nPath A: {path_a}\nPath B: {path_b}")
else:
    print(f"Loading Checkpoint A ({label_a}) from {path_a}...")
    net_a, state_a = _load_gate_net(path_a)

    print(f"Loading Checkpoint B ({label_b}) from {path_b}...")
    net_b, state_b = _load_gate_net(path_b)

    print(f"\nStarting Match: {label_a} vs {label_b} (SPRT cap {TRAIN['gate_games']} games, {TRAIN['gate_sims']} sims)...")

    metrics = play_match(
        net_a, net_b, cfg,
        n_games=TRAIN['gate_games'],
        sims=TRAIN['gate_sims'],
        device=DEVICE,
        exploration_moves=TRAIN['gate_exploration_moves'],
        tablebase=tablebase,
        sprt=True,
        sprt_elo0=ELO0,
        sprt_elo1=ELO1,
        sprt_alpha=ALPHA,
        sprt_beta=BETA,
    )
    if tablebase is not None:
        tablebase.close()
    winrate = metrics['winrate']
    wins = metrics['wins_as_white'] + metrics['wins_as_black']
    losses = metrics['losses_as_white'] + metrics['losses_as_black']
    draws = metrics['draws_as_white'] + metrics['draws_as_black']

    iter_a = _checkpoint_iteration(CHECKPOINT_A, state_a)
    iter_b = _checkpoint_iteration(CHECKPOINT_B, state_b)
    _log_gate_metrics(CKPT_DIR, iter_a, iter_b, metrics, TRAIN['gate_games'])

    wdl_notation = f"+{wins} ={draws} -{losses}"

    print("\n" + "="*40)
    print(f"MATCH COMPLETED!")
    print(f"{label_a} score vs {label_b}: {winrate:.3f} [{wdl_notation}]")
    print(f"  As White: Wins: {metrics['wins_as_white']}, Losses: {metrics['losses_as_white']}, Draws: {metrics['draws_as_white']}")
    print(f"  As Black: Wins: {metrics['wins_as_black']}, Losses: {metrics['losses_as_black']}, Draws: {metrics['draws_as_black']}")
    print(f"  Mean Game Length: {metrics['mean_game_len']:.1f} plies")
    print(f"  Terminations: {metrics['terminations']}")
    print(f"  SPRT: {sprt_verdict_label(metrics['sprt_decision'])} (llr={metrics['llr']:.2f}, games={metrics['games_played']})")
    print(f"  Logged to: {os.path.join(CKPT_DIR, 'metrics_gates.csv')}")
    print(f"Result: {label_a} {sprt_verdict_label(metrics['sprt_decision'])} vs {label_b}")
    print("="*40)

In [ ]:
# 7. Plot training progress (run AFTER at least one training iteration finishes).
import os
import pandas as pd
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 5 to finish '
          '(a few minutes on the --gpu preset), then re-run this cell.')
else:
    df = pd.read_csv(path)
    if df.empty:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        df['iter'] = pd.to_numeric(df['iter'], errors='coerce')
        df = df.dropna(subset=['iter']).copy()
        if df.empty:
            print('metrics.csv has no valid iteration rows yet.')
        else:
            for col in df.columns:
                if col != 'terminations':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            if {'train_steps', 'batch_size', 'samples'}.issubset(df.columns):
                df['reuse_ratio'] = (
                    df['train_steps'] * df['batch_size'] /
                    df['samples'].replace(0, float('nan'))
                )

            gates_path = os.path.join(CKPT_DIR, 'metrics_gates.csv')
            gates_df = pd.read_csv(gates_path) if os.path.exists(gates_path) else None
            if gates_df is not None and not gates_df.empty:
                gates_df['iter'] = pd.to_numeric(gates_df['iter'], errors='coerce')
                gates_df['winrate'] = pd.to_numeric(gates_df['winrate'], errors='coerce')
                gates_df = gates_df.dropna(subset=['iter']).copy()

            df_window = df[df['iter'] <= 20].copy()
            if len(df_window) >= 10:
                entropy_ok = float(df_window['policy_entropy'].iloc[-1]) >= 1.8
                sign_acc_series = df_window['value_sign_acc'].dropna()
                value_ok = (not sign_acc_series.empty) and float(sign_acc_series.iloc[-1]) >= 0.6
                gate_rows = int((gates_df['iter'] <= 20).sum()) if gates_df is not None and not gates_df.empty else 0
                gate_ok = gate_rows >= 1
                reuse_series = df_window['reuse_ratio'].dropna() if 'reuse_ratio' in df_window.columns else pd.Series(dtype=float)
                reuse_ok = (not reuse_series.empty) and float(reuse_series.iloc[-1]) <= 20.0
                print('Early-run checks (iter <= 20):')
                print(f"  policy_entropy >= 1.8: {entropy_ok}")
                print(f"  value_sign_acc >= 0.6: {value_ok}")
                print(f"  gate rows present: {gate_ok} (rows={gate_rows})")
                print(f"  reuse_ratio <= 20x: {reuse_ok}")
            else:
                print('Early-run checks need at least 10 completed iterations.')

            fig, ax = plt.subplots(3, 3, figsize=(18, 12))
            ax = ax.flatten()

            def plot_panel(axis, cols, title, ylim=None):
                present = [c for c in cols if c in df.columns]
                if not present:
                    axis.text(0.5, 0.5, 'missing columns', ha='center', va='center')
                    axis.set_title(title)
                    axis.set_xlabel('iteration')
                    return
                df.plot(x='iter', y=present, ax=axis, title=title)
                axis.set_xlabel('iteration')
                if ylim is not None:
                    axis.set_ylim(*ylim)

            plot_panel(ax[0], ['policy_loss', 'value_loss'], 'Losses')
            plot_panel(ax[1], ['policy_entropy', 'grad_norm'], 'Entropy / Gradient norm')
            plot_panel(ax[2], ['value_sign_acc', 'policy_top1_agree'], 'Net target agreement', ylim=(0.0, 1.0))
            plot_panel(ax[3], ['decisive_rate', 'draw_rate', 'max_moves_trunc_rate'], 'Outcome mix', ylim=(0.0, 1.0))
            plot_panel(ax[4], ['mean_game_len'], 'Mean game length')

            if gates_df is None or gates_df.empty:
                ax[5].text(0.5, 0.5, 'no gate rows yet', ha='center', va='center')
                ax[5].set_title('Strength gate')
                ax[5].set_xlabel('iteration')
            else:
                gates_df.plot(
                    x='iter',
                    y='winrate',
                    marker='o',
                    ax=ax[5],
                    title='Strength gate (vs -20 iters)',
                    legend=False,
                )
                ax[5].set_xlabel('iteration')
                ax[5].axhline(0.5, color='gray', linestyle='--', linewidth=1)
                ax[5].set_ylim(0.0, 1.0)

            plot_panel(ax[6], ['learning_rate'], 'Learning rate')
            plot_panel(ax[7], ['reuse_ratio'], 'Reuse ratio (train load / fresh samples)')
            plot_panel(ax[8], ['buffer_size'], 'Replay buffer size')

            plt.tight_layout()
            plt.show()

## After training

Download the updated **`results/`** folder (at least `latest.pt` and `metrics.csv`).
Run the analysis server locally:

```bash
set IMMORTALITE_ZERO_CHECKPOINT=path\to\results\latest.pt   # Windows
python -m uvicorn server.app:app --port 8000
```

Then open http://localhost:8000/app/ .